In [48]:
import pandas as pd
from pathlib import Path

# 设置路径
remap_path = Path("../query_yelp/ground_truth_dataset")

# 读取五个重编码后的表
df_business = pd.read_json("../query_yelp/origin_dataset/business_query.json", lines=True)
df_checkin  = pd.read_json("../query_yelp/origin_dataset/checkin_query.json", lines=True)
df_review   = pd.read_json("../query_yelp/origin_dataset/review_query.json", lines=True)
df_tip      = pd.read_json("../query_yelp/origin_dataset/tip_query.json", lines=True)
df_user     = pd.read_json("../query_yelp/origin_dataset/user_query.json", lines=True)


In [49]:
df_business

,business_id,name,review_count,is_open,attributes,hours,description
0,businessid_49,Steps to Learning Montessori Preschool,8,1,"{'BusinessAcceptsCreditCards': 'True', 'WiFi':...","{'Monday': '0:0-0:0', 'Tuesday': '8:0-17:0', '...","Located at 6901 Phelps Rd in Goleta, CA, this ..."
1,businessid_47,Breeze Blow Dry Bar,81,0,"{'ByAppointmentOnly': 'False', 'BusinessAccept...","{'Monday': '7:0-18:0', 'Tuesday': '7:0-18:0', ...","Located at 9916 Clayton Rd in St. Louis, MO, t..."
2,businessid_88,Impact Guns,39,1,"{'BusinessParking': '{'garage': False, 'street...","{'Monday': '10:0-19:0', 'Tuesday': '10:0-19:0'...","Located at 11655 W Executive Dr in Boise, ID, ..."
3,businessid_41,Palms Primary Care,5,1,None,"{'Monday': '8:30-17:0', 'Tuesday': '8:30-17:0'...","Located at 1615 Pasadena Ave S, Ste 430 in Sai..."
4,businessid_33,J&Q Nails,28,1,"{'BusinessParking': '{'garage': False, 'street...","{'Monday': '9:30-19:0', 'Tuesday': '9:30-19:0'...","Located at 9655 E US Hwy 36, Unit H in Avon, I..."
...,...,...,...,...,...,...,...
95,businessid_23,Harbor Freight,8,1,None,"{'Monday': '8:0-19:0', 'Tuesday': '8:0-19:0', ...","Located at 5211 Hickory Hollow Pkwy, Ste 103 i..."
96,businessid_38,Philadelphia Hair Studio,22,0,"{'ByAppointmentOnly': 'False', 'RestaurantsPri...","{'Tuesday': '11:0-19:0', 'Wednesday': '11:0-19...","Located at 744 S 6th St in Philadelphia, PA, t..."
97,businessid_81,Fantastic Sams Cut & Color,6,1,"{'BusinessParking': '{'garage': False, 'street...","{'Monday': '8:0-21:0', 'Tuesday': '8:0-21:0', ...","Located at 36129 E Lake Rd in Palm Harbor, FL,..."
98,businessid_13,Avian Glen Winery,12,1,"{'RestaurantsPriceRange2': '2', 'RestaurantsTa...","{'Monday': '12:0-20:0', 'Tuesday': '12:0-20:0'...","Located at 3545 Almaville Rd in Smyrna, TN, th..."


In [46]:
import pandas as pd

# 读取数据
df = pd.read_json("../query_yelp/origin_dataset/business_query.json", lines=True)

def check_description(row):
    desc = row.get("description", "")
    if not isinstance(desc, str):
        return ["missing_description"]

    missing = []

    # 地址类字段严格匹配（原文必须完整出现）
    for field in ["address", "city", "state"]:
        val = str(row.get(field, ""))
        if val and val not in desc:
            missing.append(field)

    # 类别字段逐个词项匹配
    categories = row.get("categories", "")
    if isinstance(categories, str):
        category_list = [cat.strip() for cat in categories.split(",") if cat.strip()]
        for cat in category_list:
            if cat not in desc:
                missing.append(f"category:{cat}")

    return missing

# 应用检查
df["missing_fields"] = df.apply(check_description, axis=1)
failed = df[df["missing_fields"].apply(lambda x: len(x) > 0)]

# 输出结果
if failed.empty:
    print("✅ All descriptions passed — all fields and categories are present.")
else:
    print(f"❌ {len(failed)} rows failed. Missing info:")
    for idx, row in failed.iterrows():
        print(f"- Row {idx}: missing fields: {row['missing_fields']}")



❌ 1 rows failed. Missing info:
- Row 17: missing fields: ['category:Nightlife', 'category:Bars', 'category:Restaurants', 'category:Sports Bars']


In [47]:
import pandas as pd
from pathlib import Path

# 路径设置
input_path = Path("../query_yelp/origin_dataset/business_query.json")
output_path = Path("../query_yelp/origin_dataset/business_final.json")

# 读取数据
df = pd.read_json(input_path, lines=True)

# 删除四个字段
df = df.drop(columns=["address", "city", "state", "categories"]).copy()

# 保存结果
df.to_json(output_path, orient="records", lines=True, force_ascii=False)
print(f"✅ 已删除 address/city/state/categories 字段，保存到: {output_path}")



✅ 已删除 address/city/state/categories 字段，保存到: ..\query_yelp\origin_dataset\business_final.json


In [44]:
import pandas as pd
from pathlib import Path

import json
from tqdm import tqdm
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",  
)

deployment_name = "gpt-4o-mini" 
input_path = Path("../query_yelp/origin_dataset/business_cleaned.json")
output_path = Path("../query_yelp/origin_dataset/business_query.json")

df = pd.read_json(input_path, lines=True)
def generate_description(address, city, state, categories):
    fields = {
        "address": address,
        "city": city,
        "state": state,
        "categories": categories
    }

    # 检查是否至少有一个字段非空
    if not any(fields.values()):
        return None

    # 构建 location 描述部分
    location_parts = []
    for key in ["address", "city", "state"]:
        if fields[key]:
            location_parts.append(f"'{fields[key]}'")

    location_text = ", ".join(location_parts)
    location_clause = f"located at {location_text}" if location_text else ""

    # 构建类别部分
    if categories:
        category_clause = (
                f"The sentence must include the exact list of categories as a phrase: '{categories}', "
                "with each category included verbatim and unchanged."
            )

    else:
        category_clause = ""

    # 构建完整 prompt
    prompt = (
            f"Write a single, natural-sounding English sentence that describes a business {location_clause}. "
            f"{category_clause} "
            "Do not mention the business name. Use a helpful, informative, but concise tone."
        )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant writing business location descriptions."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=150
        )
        return response.choices[0].message.content.strip()
    except Exception as e:
        print(f"❌ Error generating for {location_text}: {e}")
        return None

# ========== 主循环 ==========
descriptions = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    desc = generate_description(
        address=row.get("address", ""),
        city=row.get("city", ""),
        state=row.get("state", ""),
        categories=row.get("categories", "")
    )
    descriptions.append(desc)

# ========== 写入 ==========
df["description"] = descriptions
df.to_json(output_path, orient="records", lines=True, force_ascii=False)
print(f"✅ 完整描述生成完毕，已保存至：{output_path}")

100%|██████████| 100/100 [01:33<00:00,  1.07it/s]

✅ 完整描述生成完毕，已保存至：..\query_yelp\origin_dataset\business_query.json


In [12]:
# 删除指定列
df_business_cleaned = df_business.drop(columns=["postal_code", "latitude", "longitude", "stars"]).copy()

# 保存到 origin_dataset 文件夹
output_path = Path("../query_yelp/origin_dataset")
output_path.mkdir(parents=True, exist_ok=True)
df_business_cleaned.to_json(output_path / "yelp_business_cleaned.json", orient="records", lines=True, force_ascii=False)

print("✅ df_business 已清洗并保存至 origin_dataset/yelp_business_cleaned.json")


✅ df_business 已清洗并保存至 origin_dataset/yelp_business_cleaned.json


In [32]:
df_checkin

,business_id,date
0,businessid_2,"2011-03-18 21:32:32, 2011-07-03 19:19:32, 2011..."
1,businessid_5,"2014-01-02 16:36:55, 2017-02-23 20:43:38, 2017..."
2,businessid_6,"2010-05-16 05:52:45, 2010-07-14 17:56:05, 2010..."
3,businessid_7,"2019-12-07 06:10:53, 2020-01-01 23:40:41, 2020..."
4,businessid_8,"2012-12-07 18:35:12, 2012-12-08 22:03:39, 2012..."
...,...,...
85,businessid_94,"2011-08-04 17:42:21, 2011-08-15 03:46:17, 2012..."
86,businessid_95,"2011-04-20 23:34:07, 2011-05-05 02:29:31, 2011..."
87,businessid_96,"2010-05-08 01:48:04, 2010-05-16 02:13:28, 2010..."
88,businessid_97,"2012-03-12 14:26:10, 2013-12-09 22:30:34"


In [33]:
df_review

,review_id,user_id,business_ref,rating,useful,funny,cool,text,date
0,reviewid_135,userid_548,businessref_34,2,0,0,0,"Sure, it's cheap, but there isn't much to see....",2016-08-01 03:44:00
1,reviewid_1067,userid_213,businessref_89,5,2,0,0,Very good service but a little pricey for the ...,2021-06-14 11:39:00
2,reviewid_871,userid_616,businessref_82,4,0,0,0,My friend and I enjoyed a fantastic meal at Mi...,2013-05-29 23:01:00
3,reviewid_314,userid_1903,businessref_66,2,1,2,1,This location is not one of my favorites peopl...,2016-05-21 18:48:52
4,reviewid_487,None,businessref_95,1,0,0,0,Terrible service. I was charged twice for onli...,2021-11-01 17:11:59
...,...,...,...,...,...,...,...,...,...
1995,reviewid_687,None,businessref_16,2,5,0,0,The Fox has gone downhill since I last visited...,2014-02-06 16:08:41
1996,reviewid_202,userid_428,businessref_16,4,0,0,0,Food was great as well as the service. Big scr...,2015-02-24 06:44:00
1997,reviewid_1362,userid_1684,businessref_85,5,0,0,0,Excellent customer service. Our order took 10 ...,2017-12-03 03:37:00
1998,reviewid_1352,userid_13,businessref_96,5,2,0,1,"SO GOOD! I already love this place, even if I ...",2011-03-08 23:20:00


In [17]:
import pandas as pd
import numpy as np
from pathlib import Path

# 随机覆盖 date 字段为自然语言格式之一
def randomize_date_column(df, date_col):
    dt = pd.to_datetime(df[date_col])
    
    fmt1 = dt.dt.strftime("%Y-%m-%d %H:%M:%S")        # ISO 风格
    fmt2 = dt.dt.strftime("%B %d, %Y at %I:%M %p")    # 美式自然语言
    fmt3 = dt.dt.strftime("%d %b %Y, %H:%M")          # 紧凑自然语言
    
    all_choices = pd.DataFrame({"f1": fmt1, "f2": fmt2, "f3": fmt3})
    df[date_col] = all_choices.apply(lambda row: np.random.choice(row.values), axis=1)
    return df

# 路径设置
remap_path = Path("../query_yelp/ground_truth_dataset")
output_path = Path("../query_yelp/origin_dataset")
df_review = pd.read_json(remap_path / "yelp_review_remapped.json", lines=True)

# 重命名 stars → rating，随机格式化 date 列
df_review_cleaned = df_review.rename(columns={"stars": "rating"}).copy()
df_review_cleaned = randomize_date_column(df_review_cleaned, "date")

# 保存为 review_query.json
output_path.mkdir(parents=True, exist_ok=True)
df_review_cleaned.to_json(output_path / "review_query.json", orient="records", lines=True, force_ascii=False)

print("✅ 已将 date 列覆盖为三种自然语言格式之一并保存为 review_query.json")



✅ 已将 date 列覆盖为三种自然语言格式之一并保存为 review_query.json


In [34]:
df_tip

,user_id,business_ref,text,date,compliment_count
0,None,businessref_85,Great customer service. Great job !!!,2016-04-28 19:31:24,0
1,userid_965,businessref_12,Great place and some of the friendliest people...,2013-12-04 02:46:01,0
2,userid_909,businessref_96,Update: Blue Plate Specials will only be one M...,2015-06-23 00:22:09,0
3,None,businessref_45,Great produce section with competitive prices ...,2013-02-22 02:17:41,0
4,userid_1621,businessref_47,I know people come here for the blowouts - whi...,2015-09-19 18:58:40,0
...,...,...,...,...,...
779,userid_117,businessref_46,Best. Food. Ever. Also get the Dirty Bastard.,2016-03-09 01:20:58,0
780,userid_979,businessref_46,Love some trashed wings with a side of popover...,2020-03-21 01:31:11,0
781,None,businessref_27,This place is trash... never again...,2021-07-18 04:27:55,0
782,userid_10,businessref_8,Sign up and get $20 off your first uber ride! ...,2014-02-22 04:34:13,0


In [35]:
df_user

,user_id,name,review_count,yelping_since,useful,funny,cool,elite
0,userid_286,Todd,376,2009-01-15 16:40:06,1373,723,639,"2010,2011,2012,2013,2014"
1,userid_1331,Patt,1028,2010-07-13 15:42:03,9050,3249,5929,"2011,2012,2013,2014,2015,2016,2017,2018,2019,2..."
2,userid_1880,Norma,57,2010-09-07 23:24:36,217,57,115,"2012,2013"
3,userid_271,Antony,49,2011-10-23 19:47:29,116,159,34,
4,userid_534,Mandy,754,2011-08-30 13:46:26,2925,775,988,"2011,2012,2013,2014,2015,2016,2017,2018,2019,2..."
...,...,...,...,...,...,...,...,...
1994,userid_1023,Viet,1,2021-01-23 21:24:23,1,0,0,
1995,userid_1270,Deb,2,2012-10-03 23:10:28,5,0,0,
1996,userid_744,Debbie,2,2016-04-28 18:10:10,1,0,0,
1997,userid_818,Erica,1,2012-02-03 18:30:27,0,0,0,


In [29]:
import pandas as pd
from pathlib import Path


# 从 business 表构造 business_map：原始 business_id → businessid_N
business_ids = df_business["business_id"].tolist()  # 例：['businessid_1', ..., 'businessid_100']
original_ids = df_business.index.tolist() if df_business.index.name == 'business_id' else df_business["business_id"].index.tolist()
business_map = {original_id: business_id for original_id, business_id in zip(df_business["business_id"], business_ids)}

# 构建 business_ref_map，保持编号一致，只换前缀
business_ref_map = {
    original_id: business_id.replace("businessid_", "businessref_")
    for original_id, business_id in business_map.items()
}

# 替换 review 和 tip 中的 business_id
df_review = df_review.rename(columns={"business_id": "business_ref"}).copy()
df_review["business_ref"] = df_review["business_ref"].map(business_ref_map)

df_tip = df_tip.rename(columns={"business_id": "business_ref"}).copy()
df_tip["business_ref"] = df_tip["business_ref"].map(business_ref_map)

# 保存为新文件
output_path.mkdir(parents=True, exist_ok=True)
df_review.to_json(output_path / "review_query_rename.json", orient="records", lines=True, force_ascii=False)
df_tip.to_json(output_path / "tip_query_rename.json", orient="records", lines=True, force_ascii=False)

print("✅ df_review 和 df_tip 已保存为 review_query.json 与 tip_query.json")

# === 映射检查：打印前10个 business_id ↔ business_ref 对照 ===
print("\n📋 映射检查：原始 business_id 对应 businessid_* 和 businessref_*")
for i, (orig, bid) in enumerate(business_map.items()):
    if i >= 10: break
    ref = business_ref_map.get(orig, "❌ not mapped")
    print(f"原始ID: {orig} → business_id: {bid} → business_ref: {ref}")


✅ df_review 和 df_tip 已保存为 review_query.json 与 tip_query.json

📋 映射检查：原始 business_id 对应 businessid_* 和 businessref_*
原始ID: businessid_49 → business_id: businessid_49 → business_ref: businessref_49
原始ID: businessid_47 → business_id: businessid_47 → business_ref: businessref_47
原始ID: businessid_88 → business_id: businessid_88 → business_ref: businessref_88
原始ID: businessid_41 → business_id: businessid_41 → business_ref: businessref_41
原始ID: businessid_33 → business_id: businessid_33 → business_ref: businessref_33
原始ID: businessid_74 → business_id: businessid_74 → business_ref: businessref_74
原始ID: businessid_92 → business_id: businessid_92 → business_ref: businessref_92
原始ID: businessid_64 → business_id: businessid_64 → business_ref: businessref_64
原始ID: businessid_52 → business_id: businessid_52 → business_ref: businessref_52
原始ID: businessid_29 → business_id: businessid_29 → business_ref: businessref_29


In [30]:
import pandas as pd
from datetime import datetime
import random
# 路径配置
input_path = "../amazonreview_query\origin_dataset/review_query.json"
output_path = "../amazonreview_query\origin_dataset/review_query.json"

# ====== 时间格式化函数 ======
def format_timestamp_randomly(ts_ms):
    try:
        dt = datetime.fromtimestamp(ts_ms / 1000)
        formats = [
            dt.strftime("%Y-%m-%d %H:%M:%S"),           # ISO 风格
            dt.strftime("%B %d, %Y at %I:%M %p"),       # 美式自然语言
            dt.strftime("%d %b %Y, %H:%M")              # 英式简洁风格
        ]
        return random.choice(formats)
    except Exception as e:
        print(f"❌ 时间转换错误: {e}")
        return ts_ms  # 如果时间格式错误，保留原始值

# ====== 读取数据 ======
df = pd.read_json(input_path, lines=True, convert_dates=False)

# ====== 转换 review_time 列 ======
if "review_time" in df.columns:
    df["review_time"] = df["review_time"].apply(
        lambda ts: format_timestamp_randomly(ts) if pd.notnull(ts) else ts
    )
    print("✅ 'review_time' 字段已转换为自然语言格式")
else:
    print("⚠️ 'review_time' 字段不存在，未做处理")

# ====== 保存结果 ======
df.to_json(output_path, orient="records", lines=True)
print(f"✅ 转换后的文件已保存到：{output_path}")


✅ 'review_time' 字段已转换为自然语言格式
✅ 转换后的文件已保存到：../amazonreview_query\origin_dataset/review_query.json


In [23]:
print(df["timestamp"].iloc[0])
print(type(df["timestamp"].iloc[0]))


November 25, 2012 at 02:52 AM
<class 'str'>


In [33]:
print(df_review.dtypes)
print(df_meta.dtypes)

rating                int64
title                object
text                 object
parent_asin          object
review_time          object
helpful_vote          int64
verified_purchase      bool
dtype: object
title             object
subtitle          object
author            object
rating_number      int64
features          object
description       object
price            float64
store             object
categories        object
details           object
parent_asin       object
dtype: object


In [13]:
import json
import random
from tqdm import tqdm
from openai import AzureOpenAI

# ====== Azure OpenAI 配置 ======
client = AzureOpenAI(
    api_key="609ced4d971240b8a08f7fb0e6d846ea",
    api_version="2024-08-01-preview",
    azure_endpoint="https://promptdelta-nc.openai.azure.com",  # 不要加 /v1
)

deployment_name = "gpt-4o-mini"  # 在 Azure Portal 的 Deployments 中查看

# ====== 文件路径 ======
input_path = "../amazonreview_query\origin_dataset/light_meta_edited.json"
output_path = "../amazonreview_query\origin_dataset/books_info.json"



# ========== 细节改写函数 ==========
def enrich_details_with_llm(details_dict):
    """使用 LLM 将 details 字典改写成自然语言描述"""

    if not isinstance(details_dict, dict):
        return ""

    # 只保留非空字段
    filtered = {k: v for k, v in details_dict.items() if v and v != "null"}

    if not filtered:
        return ""

    prompt = (
        f"The following dictionary describes a book's details:\n{json.dumps(filtered, indent=2)}\n\n"
        "Please rewrite this dictionary into a single natural English paragraph. "
        "Include all factual information, do not omit any values, and do not mention the keys explicitly like 'Publisher:'. "
        "Use fluent, professional English."
    )

    try:
        response = client.chat.completions.create(
            model=deployment_name,
            messages=[
                {"role": "system", "content": "You are a helpful assistant that rewrites book metadata into natural English sentences."},
                {"role": "user", "content": prompt}
            ],
            temperature=0.7,
            max_tokens=150
        )
        return response.choices[0].message.content.strip()

    except Exception as e:
        print(f"❌ Error enriching details: {e}")
        return ""

# ========== 主处理流程 ==========
with open(input_path, "r", encoding="utf-8") as fin, \
     open(output_path, "w", encoding="utf-8") as fout:

    for line in tqdm(fin, desc="📚 Processing rows"):
        try:
            entry = json.loads(line)

            details = entry.get("details", {})
            rewritten_details = enrich_details_with_llm(details)
            entry["details"] = rewritten_details

            fout.write(json.dumps(entry, ensure_ascii=False) + "\n")

        except Exception as e:
            print(f"❌ Skipped a line due to error: {e}")
            continue

print(f"\n✅ Finished writing to: {output_path}")



📚 Processing rows: 200it [06:26,  1.93s/it]


✅ Finished writing to: ../amazonreview_query\origin_dataset/books_info.json


In [3]:
import json
import re

file_path = "../query_bookreview\original_dataset/books_info.json"

english_and_year_lines = []

with open(file_path, "r", encoding="utf-8") as f:
    for i, line in enumerate(f, 1):
        try:
            entry = json.loads(line)
            details_text = entry.get("details", "")

            # 检查是否包含 "english"（大小写不敏感）
            contains_english = bool(re.search(r'\benglish\b', details_text, re.IGNORECASE))

            # 搜索任意四位年份（1900–2099）
            match = re.search(r'\b(19|20)\d{2}\b', details_text)
            has_year = bool(match)
            year = match.group(0) if match else None

            if contains_english and has_year:
                print(f"✅ Line {i}: 包含 English，年份为 {year}")
                english_and_year_lines.append(entry)

        except Exception as e:
            print(f"❌ Line {i} 解析出错: {e}")


✅ Line 1: 包含 English，年份为 2004
✅ Line 2: 包含 English，年份为 1996
✅ Line 3: 包含 English，年份为 2012
✅ Line 4: 包含 English，年份为 2013
✅ Line 5: 包含 English，年份为 2014
✅ Line 6: 包含 English，年份为 2021
✅ Line 8: 包含 English，年份为 2015
✅ Line 9: 包含 English，年份为 2019
✅ Line 10: 包含 English，年份为 2004
✅ Line 11: 包含 English，年份为 1993
✅ Line 12: 包含 English，年份为 2022
✅ Line 13: 包含 English，年份为 2023
✅ Line 14: 包含 English，年份为 2019
✅ Line 15: 包含 English，年份为 2000
✅ Line 16: 包含 English，年份为 1997
✅ Line 18: 包含 English，年份为 2012
✅ Line 19: 包含 English，年份为 2013
✅ Line 20: 包含 English，年份为 2003
✅ Line 21: 包含 English，年份为 1945
✅ Line 22: 包含 English，年份为 2006
✅ Line 23: 包含 English，年份为 2021
✅ Line 24: 包含 English，年份为 1939
✅ Line 26: 包含 English，年份为 1995
✅ Line 28: 包含 English，年份为 1995
✅ Line 29: 包含 English，年份为 2017
✅ Line 30: 包含 English，年份为 2016
✅ Line 31: 包含 English，年份为 2012
✅ Line 32: 包含 English，年份为 1998
✅ Line 33: 包含 English，年份为 2016
✅ Line 34: 包含 English，年份为 1983
✅ Line 35: 包含 English，年份为 2006
✅ Line 36: 包含 English，年份为 2018
✅ Line 37: 包含 En

In [18]:
import pandas as pd
import random
from datetime import datetime

# ====== 路径配置 ======
input_path = "../amazonreview_query\origin_dataset\light_review_edited.json"
output_path = "../amazonreview_query\origin_dataset/review_query.json"

# ====== 时间格式化函数 ======
def format_timestamp_randomly(ts_string):
    try:
        dt = pd.to_datetime(ts_string)  # 将字符串转换为 datetime 对象
        formats = [
            dt.strftime("%Y-%m-%d %H:%M:%S"),           # ISO 风格
            dt.strftime("%B %d, %Y at %I:%M %p"),       # 美式自然语言
            dt.strftime("%d %b %Y, %H:%M"),             # 英式简洁风格
        ]
        return random.choice(formats)
    except Exception as e:
        print(f"❌ 时间格式错误: {ts_string}, error: {e}")
        return ts_string  # 如果解析失败，保留原始字符串

# ====== 加载数据 ======
df = pd.read_json(input_path, lines=True)


# ====== 转换 timestamp 字段 ======
if "timestamp" in df.columns:
    df["timestamp"] = df["timestamp"].apply(lambda ts: format_timestamp_randomly(ts) if pd.notnull(ts) else ts)
    print("✅ 'timestamp' 字段已转换为自然语言格式")
else:
    print("⚠️ 'timestamp' 字段不存在，未做处理")


# ====== 保存结果 ======
df.to_json(output_path, orient="records", lines=True, force_ascii=False)
print(f"✅ 文件已保存到：{output_path}")


✅ 'timestamp' 字段已转换为自然语言格式
✅ 文件已保存到：../amazonreview_query\origin_dataset/review_query.json


In [7]:
import pandas as pd

# 读取 JSON 文件
df_meta = pd.read_json("../query_bookreview\original_dataset/books_info.json", lines=True)
df_review = pd.read_json("../query_bookreview\original_dataset/review_query.json", lines=True, convert_dates=False)

# Step 1: 获取唯一的 parent_asin
unique_asins = df_meta['parent_asin'].dropna().unique()

# Step 2: 创建 book_id 和 purchase_id 映射
asin_to_bookid = {asin: f'bookid_{i+1}' for i, asin in enumerate(unique_asins)}
asin_to_purchaseid = {asin: f'purchaseid_{i+1}' for i, asin in enumerate(unique_asins)}

# Step 3: 替换 df_meta 中的 parent_asin 为 book_id
df_meta['book_id'] = df_meta['parent_asin'].map(asin_to_bookid)
df_meta = df_meta.drop(columns=['parent_asin'])

# Step 4: 替换 df_review 中的 parent_asin 为 purchase_id
df_review['purchase_id'] = df_review['parent_asin'].map(asin_to_purchaseid)
df_review = df_review.drop(columns=['parent_asin'])

# Step 5: 创建映射表：asin → book_id + purchase_id
mapping_df = pd.DataFrame({
    'asin': unique_asins,
    'book_id': [asin_to_bookid[asin] for asin in unique_asins],
    'purchase_id': [asin_to_purchaseid[asin] for asin in unique_asins],
})

# 显示映射表前几行
print(mapping_df.head(10))

df_meta.to_json("../query_bookreview\original_dataset/books_info_remapped.json", orient='records', lines=True)

# 保存修改后的 df_review
df_review.to_json("../query_bookreview\original_dataset/review_query_remapped.json", orient='records', lines=True)


         asin    book_id    purchase_id
0  0701169850   bookid_1   purchaseid_1
1  0435088688   bookid_2   purchaseid_2
2  0316185361   bookid_3   purchaseid_3
3  0545425573   bookid_4   purchaseid_4
4  B00KFOP3RG   bookid_5   purchaseid_5
5  B09PHG4FQ8   bookid_6   purchaseid_6
6  B0086HQWC4   bookid_7   purchaseid_7
7  1680450263   bookid_8   purchaseid_8
8  1694621731   bookid_9   purchaseid_9
9  1932225323  bookid_10  purchaseid_10
